# Order Flow Imbalance

In [2]:
# imports
import pandas as pd
import numpy as np

In [3]:
# Read in the CVS file
df = pd.read_csv('first_25000_rows.csv')

## Cleaning the Data

In [29]:
df.head()

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol,timestamp
0,2024-10-21T11:54:29.221230963Z,2024-10-21T11:54:29.221064336Z,10,2,38,C,B,1,233.62,2,...,1,7,233.25,234.13,55,400,2,1,AAPL,2024-10-21 11:54:29.221064336+00:00
1,2024-10-21T11:54:29.223936626Z,2024-10-21T11:54:29.223769812Z,10,2,38,A,B,0,233.67,2,...,1,7,233.25,234.13,55,400,2,1,AAPL,2024-10-21 11:54:29.223769812+00:00
2,2024-10-21T11:54:29.225196809Z,2024-10-21T11:54:29.225030400Z,10,2,38,A,B,0,233.67,3,...,1,7,233.25,234.13,55,400,2,1,AAPL,2024-10-21 11:54:29.225030400+00:00
3,2024-10-21T11:54:29.712600612Z,2024-10-21T11:54:29.712434212Z,10,2,38,A,B,2,233.52,200,...,1,7,233.25,234.13,55,400,2,1,AAPL,2024-10-21 11:54:29.712434212+00:00
4,2024-10-21T11:54:29.764839221Z,2024-10-21T11:54:29.764673165Z,10,2,38,C,B,2,233.52,200,...,1,7,233.25,234.13,55,400,2,1,AAPL,2024-10-21 11:54:29.764673165+00:00


In [46]:
df.tail()

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol,timestamp
4995,2024-10-21T13:04:16.583694069Z,2024-10-21T13:04:16.583527688Z,10,2,38,A,B,2,233.46,200,...,1,2,233.25,234.50,55,63,2,4,AAPL,2024-10-21 13:04:16.583527688+00:00
4996,2024-10-21T13:04:17.976627074Z,2024-10-21T13:04:17.976461017Z,10,2,38,A,A,1,233.69,200,...,1,2,233.25,234.50,55,63,2,4,AAPL,2024-10-21 13:04:17.976461017+00:00
4997,2024-10-21T13:04:20.085804687Z,2024-10-21T13:04:20.085638629Z,10,2,38,C,B,2,233.46,200,...,2,2,233.24,234.50,1,63,1,4,AAPL,2024-10-21 13:04:20.085638629+00:00
4998,2024-10-21T13:04:20.085817362Z,2024-10-21T13:04:20.085651109Z,10,2,38,A,B,3,233.44,200,...,1,2,233.25,234.50,55,63,2,4,AAPL,2024-10-21 13:04:20.085651109+00:00
4999,2024-10-21T13:04:20.131008559Z,2024-10-21T13:04:20.130842270Z,10,2,38,A,A,0,233.56,14,...,1,1,233.25,234.43,55,105,2,2,AAPL,2024-10-21 13:04:20.130842270+00:00


In [77]:
df['symbol'].unique()

array(['AAPL'], dtype=object)

In [7]:
# Understanding the data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 74 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ts_recv        5000 non-null   object 
 1   ts_event       5000 non-null   object 
 2   rtype          5000 non-null   int64  
 3   publisher_id   5000 non-null   int64  
 4   instrument_id  5000 non-null   int64  
 5   action         5000 non-null   object 
 6   side           5000 non-null   object 
 7   depth          5000 non-null   int64  
 8   price          5000 non-null   float64
 9   size           5000 non-null   int64  
 10  flags          5000 non-null   int64  
 11  ts_in_delta    5000 non-null   int64  
 12  sequence       5000 non-null   int64  
 13  bid_px_00      5000 non-null   float64
 14  ask_px_00      5000 non-null   float64
 15  bid_sz_00      5000 non-null   int64  
 16  ask_sz_00      5000 non-null   int64  
 17  bid_ct_00      5000 non-null   int64  
 18  ask_ct_0

In [26]:
# Creating a timestap column from ts_event
df['timestamp'] = pd.to_datetime(df['ts_event'])

## Best-Level OFI

In [22]:
def filter_order_book_data(data,asset,t_input,h_input):
    """
    Returns filtered df for a single asset and timeframe, sorted by timestamp.

    Args:
        data (pd.DataFrame): The original dataframe.
        asset (str): The symbol of the asset.
        t_input (pd.Timestamp): The end time of the interval.
        h_input (pd.Timedelta): The duration of the interval.

    Returns:
        pd.DataFrame: Filtered DataFrame sorted by timestamp, or an empty DataFrame if no data is found.
    """
    
    # Filter for only the single asset
    asset_df = data[data['symbol'] == asset].copy()

    # Calculate the start and end time
    start_time = t_input-h_input
    end_time = t_input

    # Filter for timestamp
    filtered_df = asset_df[(asset_df['timestamp'] > start_time) & (asset_df['timestamp'] <= end_time)]

    # Sort and reset index
    filtered_df = filtered_df.sort_values(by='timestamp').reset_index(drop=True)

    return filtered_df


In [23]:
def best_level_ofi(data,asset,t_input,h_input):
    """
    Best-Level OFI Function produces the best-level (00) order flow imbalance
    for a single stock over the time interval (t-h, t].

    Args:
        data (pd.DataFrame): The original dataframe.
        asset (str): The symbol of the asset.
        t_input (pd.Timestamp): The end time of the interval.
        h_input (pd.Timedelta): The duration of the interval.

    Returns:
        float or np.nan: The calculated Best-Level OFI, or NaN if insufficient data.
    """

    if not isinstance(t_input, pd.Timestamp):
        try:
            t = pd.to_datetime(t_input)
        except Exception as e:
            print(f"Error converting end time {t_input} to datetime: {e}")
            return pd.DataFrame() # Return empty dataframe on error
    else:
        t = t_input

    # Check h for the right timedelta format
    if not isinstance(h_input, pd.Timedelta):
        try:
            h = pd.to_timedelta(h_input)
        except Exception as e:
            print(f"Error converting interval {h_input} to timedelta: {e}")
            return pd.DataFrame() # Return empty dataframe on error
    else:
        h = h_input
    
    # Return the filtered data frame
    ob_df = filter_order_book_data(data,asset,t,h)

    # Check if there's enough data after filtering
    if len(ob_df) < 2:
        # Cannot calculate OFI if there are fewer than 2 events in the interval
        print(f"Warning: Not enough data ({len(ob_df)} events) for OFI calculation for {asset} ending at {t} with interval {h}.")
        return np.nan # Return NaN to indicate calculation was not possible

    # Initialize order flow imbalance as 0
    ofi = 0

    for n in range(1,len(ob_df)):
        
        # Get the current and previous rows using iloc
        current_row = ob_df.iloc[n]
        prev_row = ob_df.iloc[n-1]

        # Get the best level (level 00) price and size data for current and previous rows
        current_bid_px = current_row['bid_px_00']
        current_ask_px = current_row['ask_px_00']
        current_bid_sz = current_row['bid_sz_00']
        current_ask_sz = current_row['ask_sz_00']

        prev_bid_px = prev_row['bid_px_00']
        prev_ask_px = prev_row['ask_px_00']
        prev_bid_sz = prev_row['bid_sz_00']
        prev_ask_sz = prev_row['ask_sz_00']

        # Compute Bid Order Flow for the current event (n)
        bid_of_n = 0 # Initialize bid order flow for this event

        if current_bid_px > prev_bid_px:
            # Case 1: Bid price increased
            bid_of_n = current_bid_sz
        elif current_bid_px == prev_bid_px:
            # Case 2: Bid price is the same
            bid_of_n = current_bid_sz - prev_bid_sz
        else: # current_bid_px < prev_bid_px
            # Case 3: Bid price decreased
            bid_of_n = -(prev_bid_sz)

        # Compute Ask Order Flow for the current event (n)
        ask_of_n = 0 # Initialize ask order flow for this event

        if current_ask_px > prev_ask_px:
             # Case 1: Ask price increased
             ask_of_n = -(prev_ask_sz)
        elif current_ask_px == prev_ask_px:
            # Case 2: Ask price is the same
            ask_of_n = current_ask_sz - prev_ask_sz
        else: # current_ask_px < prev_ask_px
            # Case 3: Ask price decreased
            ask_of_n = current_ask_sz

        # Add the OFI for this event to the total OFI for the interval
        ofi += (bid_of_n - ask_of_n)

    return ofi

In [34]:
# Test case

best_level_ofi(df,'AAPL','2024-10-21 13:04:00 +00:00',pd.Timedelta(minutes=2))

-4185

## Multi-Level OFI

In [42]:
# Editing the above function to be able to take in level as an input

def single_level_ofi(asset_df,level):
    """
    Single Level OFI Function calculates the total level order flow imbalance 
    for a given book for a single stock over the time interval (t-h, t].

    Args:
        asset_df (pd.DataFrame): Asset dataframe for the time interval.
        level (int): The level of the order book.

    Returns:
        float or np.nan: The calculated total level OFI, or NaN if insufficient data.
    """

    # Initialize order flow imbalance as 0
    total_ofi = 0

    # Construct column names based on the level number
    bid_px_col = f'bid_px_{level:02d}'
    ask_px_col = f'ask_px_{level:02d}'
    bid_sz_col = f'bid_sz_{level:02d}'
    ask_sz_col = f'ask_sz_{level:02d}'

    for n in range(1,len(asset_df)):
        
        # Get the current and previous rows using iloc
        current_row = asset_df.iloc[n]
        prev_row = asset_df.iloc[n-1]

        # Get the best level (level 00) price and size data for current and previous rows
        current_bid_px = current_row[bid_px_col]
        current_ask_px = current_row[ask_px_col]
        current_bid_sz = current_row[bid_sz_col]
        current_ask_sz = current_row[ask_sz_col]

        prev_bid_px = prev_row[bid_px_col]
        prev_ask_px = prev_row[ask_px_col]
        prev_bid_sz = prev_row[bid_sz_col]
        prev_ask_sz = prev_row[ask_sz_col]

        # Compute Bid Order Flow for the current event (n)
        bid_of_n = 0 # Initialize bid order flow for this event

        if current_bid_px > prev_bid_px:
            # Case 1: Bid price increased
            bid_of_n = current_bid_sz
        elif current_bid_px == prev_bid_px:
            # Case 2: Bid price is the same
            bid_of_n = current_bid_sz - prev_bid_sz
        else: # current_bid_px < prev_bid_px
            # Case 3: Bid price decreased
            bid_of_n = -(prev_bid_sz)

        # Compute Ask Order Flow for the current event (n)
        ask_of_n = 0 # Initialize ask order flow for this event

        if current_ask_px > prev_ask_px:
             # Case 1: Ask price increased
             ask_of_n = -(prev_ask_sz)
        elif current_ask_px == prev_ask_px:
            # Case 2: Ask price is the same
            ask_of_n = current_ask_sz - prev_ask_sz
        else: # current_ask_px < prev_ask_px
            # Case 3: Ask price decreased
            ask_of_n = current_ask_sz

        # Add the OFI for this event to the total OFI for the interval
        total_ofi += (bid_of_n - ask_of_n)

    return total_ofi

In [40]:
def multi_level_ofi(data,asset,t_input,h_input,num_levels=10):
    """
    Multi-Level OFI Function calculates the multi-level OFI vector
    for a single stock over the time interval (t-h, t].

    Args:
        data (pd.DataFrame): The original dataframe.
        asset (str): The symbol of the asset.
        t_input (pd.Timestamp): The end time of the interval.
        h_input (pd.Timedelta): The duration of the interval.

    Returns:
        float or np.nan: The calculated Best-Level OFI, or NaN if insufficient data.
    """

    if not isinstance(t_input, pd.Timestamp):
        try:
            t = pd.to_datetime(t_input)
        except Exception as e:
            print(f"Error converting end time {t_input} to datetime: {e}")
            return pd.DataFrame() # Return empty dataframe on error
    else:
        t = t_input

    # Check h for the right timedelta format
    if not isinstance(h_input, pd.Timedelta):
        try:
            h = pd.to_timedelta(h_input)
        except Exception as e:
            print(f"Error converting interval {h_input} to timedelta: {e}")
            return pd.DataFrame() # Return empty dataframe on error
    else:
        h = h_input

    ob_df = filter_order_book_data(data, asset, t, h)

    if len(ob_df) < 2:
         print(f"Warning: Not enough data ({len(ob_df)} events) for Multi-Level OFI calculation for {asset} ending at {t} with interval {h}.")
         return [np.nan] * num_levels
    
    # Calculate OFI for each level
    total_ofi_per_level = []
    
    for m in range(num_levels):
        level_ofi = single_level_ofi(ob_df, level=m)
        total_ofi_per_level.append(level_ofi)
    
    # Scaling Factor of Q
    total_sum_events = 0
    delta_N = len(ob_df)

    # Check we don't divide by 0
    if delta_N > 0:
        # Iterate through the specified number of levels M
        for m in range(num_levels):
             # Column names for bid and ask size at the current level m
             bid_sz_col = f'bid_sz_{m:02d}'
             ask_sz_col = f'ask_sz_{m:02d}'

             # Look at just the level m bid and ask size columns
             bid_sz_this_level = ob_df[bid_sz_col]
             ask_sz_this_level = ob_df[ask_sz_col]

             # Iterate through each event in the filtered dataframe 
             # You could iterate through the index or use iterrows() again, or even sum the Series directly
             for n in range(len(ob_df)): # Loop through event indices
                 # Get the size for event n at level m
                 current_bid_sz = bid_sz_this_level.iloc[n]
                 current_ask_sz = ask_sz_this_level.iloc[n]

                 # Add the bid size and ask size at level 'm' for the current event to the total sum
                 total_sum_events += (current_bid_sz + current_ask_sz)

        # Calculate Q based on the formula...
        Q = (1 / (num_levels * delta_N * 2)) * total_sum_events

    else:
        # If there are no events, Q cannot be calculated
        Q = 0
        
    # ofi vector calculation
    multi_level_ofi = []

    # Iterate through each level
    for m in range(num_levels):

        # Check if Q is not zero to avoid zero division error
        if Q != 0:
            # Calculate OFI/Q for level m
            scaled_ofi_m = total_ofi_per_level[m] / Q
        else:
            # If Q is zero, return NaN
            scaled_ofi_m = np.nan 

        # Add the scaled OFI for level m to the results list
        multi_level_ofi.append(scaled_ofi_m)

    # Return the scaled OFI value for each level
    return multi_level_ofi


In [43]:
# Test case

multi_level_ofi(df,'AAPL','2024-10-21 13:04:00 +00:00',pd.Timedelta(minutes=2),num_levels=10)

[-32.897964059396216,
 -53.89449022490333,
 -21.633260953753496,
 -17.443388828626095,
 -12.152987439862974,
 -11.704914811097003,
 -7.287075909930774,
 1.4935754292198997,
 3.348753330777249,
 -0.8332578710384704]

## Integrated OFI

In [65]:
# import PCA libraries
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [45]:
def integrated_ofi(data,asset,t_input,h_input,w1_vector,num_levels=10):
    """
    Calculates the Integrated OFI for a single stock at a given timestamp and interval,
    using a first principal component vector calculated from historic data (w1_vector).

    Args:
        data (pd.DataFrame): The original dataframe.
        asset (str): The symbol of the asset.
        t_input: The end time of the interval (can be string, datetime, or Timestamp).
        h_input: The duration of the interval (can be string or Timedelta).
        w1_vector (np.ndarray or list): The first principal component vector (weights) from PCA.
        num_levels (int): The number of levels used for the w1_vector.

    Returns:
        float or np.nan: The calculated Integrated OFI, or NaN if calculation is not possible.
    """

    # Calculate the multi-level OFI vector for the current timestamp
    multi_level_ofi_vec = multi_level_ofi(data,asset,t_input,h_input,num_levels)

    # Check if the multi-level OFI calculation was successful
    if any(pd.isna(multi_level_ofi_vec)):
        print(f"Warning: Multi-Level OFI calculation failed for {asset} ending at {t_input}. Cannot calculate Integrated OFI.")
        return np.nan
    
    # Ensure w1_vector is a numpy array for dot product
    w1_vector = np.asarray(w1_vector)

    # Check that the vectors are the same length
    if len(multi_level_ofi_vec) != len(w1_vector):
        print(f"Error: Dimension mismatch between Multi-Level OFI vector ({len(multi_level_ofi_vec)}) and First Principal vector ({len(w1_vector)}).")
        return np.nan
    
    # Calculate the dot product
    dot_product = np.dot(w1_vector, multi_level_ofi_vec)

    # Calculate the L1 norm of the first principal vector
    l1_norm = np.sum(np.abs(w1_vector))

    # Check that the l1_norm is not 0 for division
    if l1_norm != 0:
        integrated_ofi_value = dot_product / l1_norm

    else:
        print("Warning: L1 norm of w1_vector is zero. Cannot calculate Integrated OFI.")
        integrated_ofi_value = np.nan
    
    return integrated_ofi_value

### PCA

#### Obtain historical data for the Mutli-Level OFI for a given asset

In [59]:
# Select an asset
training_asset = 'AAPL'

training_df = df[df['symbol']=='AAPL'].copy()

In [60]:
# Choose a training period that is prior to the timestamp requested
training_start_time = training_df['timestamp'].min()
training_end_time = training_start_time + pd.Timedelta(minutes=12)

In [61]:
current_time = training_start_time + pd.Timedelta(minutes=1)
training_ofi = []

while current_time <= training_end_time:
    ml_ofi = multi_level_ofi(df,training_asset,current_time,pd.Timedelta(minutes=1),num_levels=10)

    # Only append the vector if it does not contain any NaN values.
    if not np.isnan(ml_ofi).any():
        training_ofi.append(ml_ofi)

    # Move to the next timestamp based on the calculation frequency.
    current_time += pd.Timedelta(minutes=1)

In [62]:
# Create the collection of Multi-Level OFI vectors
training_ofi_matrix = np.vstack(training_ofi)

In [63]:
# Check to make sure that the matrix is suitable for PCA
if training_ofi_matrix.shape[0] == 0:
    print("Error: training_ofi_matrix is empty. Cannot perform PCA.")

else:
    # Check that there are more samples than features
    num_samples, num_features = training_ofi_matrix.shape
    print(f"Historical OFI matrix shape: {num_samples} samples, {num_features} features (levels)")

    if num_samples > num_features:
        print("Condition met: Number of samples is greater than the number of features.")
        can_perform_pca = True
    else:
        print("Error: Number of samples is not greater than the number of features.")
        print("       PCA requires more samples than features for meaningful results.")
        can_perform_pca = False

Historical OFI matrix shape: 12 samples, 10 features (levels)
Condition met: Number of samples is greater than the number of features.


#### Perform PCA: Standardization and First Component Vector

In [68]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Standardize the matrix
standardized_ofi_matrix = scaler.fit_transform(training_ofi_matrix)
standardized_ofi_matrix.shape

(12, 10)

In [70]:
# Initialize PCA and only interested in the first component
pca = PCA(n_components=1)

# Fit PCA to standardized data
pca.fit(standardized_ofi_matrix)

# Extract the first component
w1 = pca.components_[0]
w1,w1.shape

(array([ 0.16399156,  0.15265827, -0.01550615, -0.04994346,  0.39419157,
         0.34353419,  0.44259454,  0.42084127,  0.40617528,  0.36835993]),
 (10,))

#### Run Integrated OFI Function

In [72]:
integrated_ofi(df,training_asset,(training_end_time+pd.Timedelta(minutes=4)),pd.Timedelta(minutes=2),w1,num_levels=10)

2.6241713444547448

## Cross-Asset OFI

In [74]:
def cross_asset_ofi(data, primary_asset, t_input, h_input, other_assets_list):
    """
    Calculates the Cross-Asset OFI feature as the sum of Best-Level OFI
    for a list of other assets for a given timestamp and interval (Section 3.1.3).

    Args:
        data (pd.DataFrame): The original dataframe.
        primary_asset (str): The symbol of the main asset for which Cross-Asset OFI is calculated.
        t_input: The end time of the interval (can be string, datetime, or Timestamp).
        h_input: The duration of the interval (can be string or Timedelta).
        other_assets_list (list of str): A list of symbols for the other assets to include in the sum.

    Returns:
        float or np.nan: The sum of Best-Level OFI for the other assets,
                         or NaN if the calculation for any included asset fails.
    """

    # Initializing the sum of all the total OFI from other assets
    total_cross_asset_ofi = 0.0

    # Loop through other_assets_list
    for j in other_assets_list:
        
        # Check to make sure the primary asset isn't in the list
        if j == primary_asset:
            continue

    j_best_level_ofi = best_level_ofi(data,j,t_input,h_input)

    if pd.notna(j_best_level_ofi):
        total_cross_asset_ofi =+ j_best_level_ofi
    
    return total_cross_asset_ofi


In [81]:
# Test Case
primary_asset = 'AAPL'
other_assets_list = df['symbol'].unique()
other_assets_list = np.delete(primary_asset)

cross_asset_ofi(df, primary_asset, (training_end_time+pd.Timedelta(minutes=4)),pd.Timedelta(minutes=2), other_assets_list)

TypeError: delete() missing 1 required positional argument: 'obj'

Given that the csv file only has a single asset, it is not possible to run the code for multiple assets. Given a more extensive list this code will run.